# Batch Fermenter — Does Aeration Meet Oxygen Demand?

A sparged batch fermenter is the next step up from a sealed MTP well: instead of
relying on a finite headspace reservoir, fresh air flows in continuously through a
sparger, so oxygen supply is governed by the gas-liquid mass transfer rate rather
than headspace inventory. Two design questions arise: does the chosen kLa deliver
enough O₂ to keep pace with peak growth, and does CO₂ stripping allow pH to be
held by a proportional base-dosing controller?

We work through the OTR/OUR balance and derive a kLa requirement from first
principles, then state explicit predictions before building the model using
`KineticTransferModel`, `GasFeed`, `PressureReliefVent`, and `PHController`.
Each prediction is verified against the simulation.

## 2.1 Background

The MTP well (Example 1) is O₂-limited by design: the sealed headspace is a
finite reservoir and growth halts when the O₂ is exhausted. A sparged bench
fermenter breaks this constraint — growth can, in principle, proceed until
substrate is exhausted.

This introduces a new design degree of freedom: the volumetric mass-transfer
coefficient kLa, which determines whether the supplied O₂ can be absorbed fast
enough to meet the biological demand. The minimum viable kLa is found by setting
OTR = OUR at the desired dissolved-oxygen operating point.

## 2.2 Design Basis

### 2.2.1 System configuration

| Parameter | Value | Basis |
|---|---|---|
| Total volume | 2.0 L | Standard bench bioreactor |
| Liquid volume | 1.6 L (80 %) | Conventional agitator-clearance fill fraction |
| Headspace volume | 0.4 L (20 %) | |
| Temperature | 32 °C (305.15 K) | Near-optimum for PEKILO |
| Sparging | 1 vvm (air) | Typical starting point for aerobic fermentation |
| kLa (O₂) | 2100 /h | Sized from OTR/OUR balance (§2.2.6); OTR/OUR ≥ 1.5× at 50 % DO with S₀ = 20 g/L |
| pH setpoint | 6.0 | Phosphate buffer range; CO₂-driven drift resisted by H₂PO₄⁻ /HPO₄²⁻ |

### 2.2.2 Growth medium

| Parameter | Value | Role |
|---|---|---|
| Glucose ($S_0$) | 20 g/L | Sole carbon and energy source |
| PEKILO inoculum ($X_0$) | 0.1 g/L | Starting biomass |
| NH₄Cl | 5 g/L | Nitrogen source for CHNO growth balance (NH₄⁺ ⇌ NH₃, pKₐ 9.25) |
| KH₂PO₄ | 30 g/L | Phosphate buffer and phosphorus source |
| Temperature | 32 °C (305.15 K) | |
| Initial pH | 6.0 (NaOH corrected) | Set before simulation using `cv.equilibrate_to_pH` |

KH₂PO₄ dissolves to K⁺ and H₂PO₄⁻. The H₂PO₄⁻ / HPO₄²⁻ couple (pKₐ 7.2) buffers around pH 6–7,
limiting the CO₂-driven pH drop and reducing the base dose required from the pH controller.

### 2.2.3 Biological growth model

Monod kinetics with dual-substrate limitation on glucose and dissolved O₂:

$$
\mu = \mu_{\max} \frac{S}{K_S + S} \cdot \frac{C_{O_2}}{K_{O_2} + C_{O_2}}
$$

| Parameter | Symbol | Value |
|---|---|---|
| Maximum specific growth rate | $\mu_{\max}$ | 0.5 h⁻¹ |
| Substrate half-saturation constant | $K_S$ | 5 × 10⁻³ g/L |
| O₂ half-saturation constant | $K_{O_2}$ | 0.2 × 10⁻³ g/L |
| Biomass yield on substrate | $Y$ | 0.36 g biomass / g glucose |

The PEKILO biomass formula is CH₁.₆₁O₀.₅₆N₀.₁₅ (MW ≈ 24.694 g/mol, auto-computed from atoms).
Stoichiometry is closed on **C, H, N, and O**: NH₃ — sourced from the NH₄Cl medium via
NH₄⁺ ⇌ NH₃ + H⁺ — is the nitrogen source; O₂, CO₂, and H₂O coefficients follow from the CHNO
elemental balance.

### 2.2.4 Chemical reactions

Six fast acid-base equilibria are imported from `BIOPROCESS_BASIC`:

| Reaction | pKₐ | Label | Why it matters |
|---|---|---|---|
| CO₂(aq) + H₂O ⇌ HCO₃⁻ + H⁺ | 6.35 | `eq_CO2` | Links dissolved CO₂ to pH |
| H₂O ⇌ H⁺ + OH⁻ | 14.0 | `eq_water` | Closes the proton balance |
| NH₄⁺ ⇌ NH₃(aq) + H⁺ | 9.25 | `eq_NH4` | N-source speciation; NH₃ consumed in CHNO stoichiometry |
| H₃PO₄ ⇌ H₂PO₄⁻ + H⁺ | 2.15 | `eq_phosphate_1` | Phosphate ladder (lower rung) |
| H₂PO₄⁻ ⇌ HPO₄²⁻ + H⁺ | 7.20 | `eq_phosphate_2` | Dominant buffer near pH 6–7 |
| HPO₄²⁻ ⇌ PO₄³⁻ + H⁺ | 12.35 | `eq_phosphate_3` | Phosphate ladder (upper rung) |

Initial medium is loaded in ionic form (NH₄⁺ + Cl⁻ for NH₄Cl; K⁺ + H₂PO₄⁻ for KH₂PO₄);
no salt-dissolution reactions are needed.

### 2.2.5 Gas-liquid mass transfer

`KineticTransferModel` is used for O₂, CO₂, and NH₃; `EquilibriumTransferModel` for N₂:

| Model | Species | Reason |
|---|---|---|
| `KineticTransferModel` | O₂, CO₂, NH₃ | Transfer rate is comparable to biological demand |
| `EquilibriumTransferModel` | N₂ | Biologically inert; fast equilibration adequate |

CO₂ and NH₃ use `transfer_basis="molecular"` so transfer is driven by the dissolved molecular
forms (CO₂(aq) and NH₃(aq)), ensuring carbonate and ammonia equilibria determine speciation
rather than the transfer model.

### 2.2.6 Oxygen budget

The critical check is OTR ≥ OUR. Both are evaluated at 50% dissolved oxygen saturation —
a conservative operating point:

$$
OTR = k_La \cdot \tfrac{1}{2}C^*_{O_2} \cdot V_L, \qquad
OUR_{\max} = \nu_{O_2} \cdot \frac{\mu_{\max} \cdot X_{\max}}{Y \cdot \text{MW}_S} \cdot V_L
$$

where $\nu_{O_2}$ is derived from the **CHNO elemental balance** (same formula as §1.2.6)
and $X_{\max} \approx Y \cdot S_0 + X_0$.

In [ ]:
# CHNO stoichiometric coefficients (per mol glucose consumed)
# Glucose C6H12O6; PEKILO CH1.61O0.56N0.15; N-source NH3 (no O atoms)
import math
R_ATM   = 0.08205   # L·atm / mol·K
T_K_DB  = 305.15
MW_S_DB = 180.156   # g/mol  Glucose
MW_X_DB = 12.011 + 1.61*1.008 + 0.56*15.999 + 0.15*14.007
Y_DB    = 0.36

Y_mol  = Y_DB * MW_S_DB / MW_X_DB
nu_CO2 = 6 - Y_mol
nu_NH3 = Y_mol * 0.15
nu_H2O = (12 + nu_NH3*3 - Y_mol*1.61) / 2
nu_O2  = (Y_mol*0.56 + nu_CO2*2 + nu_H2O - 6) / 2

print(f"nu_O2  = {nu_O2:.3f} mol O2 / mol glucose  (CHNO balance)")
print(f"MW_X   = {MW_X_DB:.3f} g/mol  (auto from atoms)")

# Vessel
V_LIQ_DB = 1.6    # L

# O2 saturation at headspace conditions (~305 K, 1 atm air)
H_O2   = 1.3e-5   # mol/(m3·Pa) Henry constant at ~305 K
y_O2   = 0.21
C_star = H_O2 * y_O2 * 101325 / 1000  # mol/L  (÷1000 converts m³→L)

# Peak OUR at S0 = 20 g/L
MU_MAX_DB = 0.5; YIELD_DB = 0.36; S0_DB = 20.0; X0_DB = 0.1
X_peak    = YIELD_DB * S0_DB + X0_DB
OUR_vol   = nu_O2 * MU_MAX_DB * X_peak / (YIELD_DB * MW_S_DB)
OUR_tot   = OUR_vol * V_LIQ_DB

# Minimum kLa for OTR/OUR ≥ 1.5× at 50 % DO saturation:
#   kLa · (0.5 · C*) ≥ 1.5 · OUR_vol   →   kLa_min = 1.5 · OUR_vol / (0.5 · C*)
kLa_min   = 1.5 * OUR_vol / (0.5 * C_star)
KLA_O2_DB = math.ceil(kLa_min / 100) * 100   # round up to nearest 100 /h

# OTR at 50 % DO with design kLa
OTR_vol = KLA_O2_DB * 0.5 * C_star
OTR_tot = OTR_vol * V_LIQ_DB

# Sparging O2 supply check
Q_gas   = 1.0 * V_LIQ_DB        # L/min  (1 vvm)
n_O2_sp = y_O2 * Q_gas * 60 / (R_ATM * T_K_DB)  # mol/h
util    = OUR_tot / n_O2_sp * 100

print(f"\nPeak OUR:       {OUR_tot*1000:.1f} mmol/h  (X_peak = {X_peak:.2f} g/L)")
print(f"C*(O2) at 305 K: {C_star*1000:.3f} mmol/L")
print(f"Min kLa(O2):    {kLa_min:.0f} /h  (1.5× OUR at 50 % DO)")
print(f"Design kLa:     {KLA_O2_DB:.0f} /h  (rounded up to nearest 100)")
print(f"\nOTR at 50 % DO: {OTR_tot*1000:.1f} mmol/h  ({OTR_vol*1000:.2f} mmol/L/h)")
print(f"OTR / OUR:      {OTR_tot/OUR_tot:.2f}×  ({'PASS' if OTR_tot/OUR_tot >= 1.5 else 'FAIL'} — target ≥ 1.5×)")
print(f"Sparging O2:    {n_O2_sp*1000:.0f} mmol/h  (1 vvm)")
print(f"O2 util:        {util:.1f}%  (excess vented by pressure relief)")

### 2.2.7 Predicted behaviour

| # | Prediction | Basis |
|---|---|---|
| 1 | **Dissolved oxygen stays above zero** throughout the run | OTR/OUR ≈ 1.5× at 50 % DO; adequate safety margin |
| 2 | **Growth is substrate-limited**, not O₂-limited | OTR safely exceeds peak OUR |
| 3 | **Endpoint biomass ≈ 7.3 g/L** | $Y \cdot S_0 + X_0 = 0.36 \times 20 + 0.1 \approx 7.3$ g/L |
| 4 | **pH tracks setpoint** with modest dosing | Phosphate buffer (H₂PO₄⁻/HPO₄²⁻, pKₐ 7.2) reduces controller excursions |

## 2.3 Bioreactor Modelling

With the design basis established, we build the model in PyOMES and verify each prediction.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


def _setup_vlsim_path():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "pyproject.toml").exists():
            if str(p / "models") not in sys.path:
                sys.path.insert(0, str(p / "models"))
            return
    raise RuntimeError("Run from inside the PyOMES repo")


_setup_vlsim_path()

from PyOMES.chemistry import Species
from PyOMES.chemistry.common_species import (
    NH4_plus as NH4_PLUS,
    H3PO4, H2PO4_minus as H2PO4_MINUS, HPO4_2minus as HPO4_2MINUS, PO4_3minus as PO4_3MINUS,
    K_plus as K_PLUS, Cl_minus as CHLORIDE, Na_plus as NA_PLUS,
)
from PyOMES.chemistry.databases.bioprocess_basic import (
    BIOPROCESS_BASIC,
    NH4Cl as NH4CL, KH2PO4 as KH2PO4_SALT,
)
from PyOMES.chemistry.databases.anaerobic_digestion import AD_BASIC
from PyOMES.reactions import ReactionBuilder, ReactionSystem
from PyOMES.core import (
    ControlVolume, EquilibriumTransferModel, KineticTransferModel,
    GasPhase, LiquidPhase, Simulation,
)
from PyOMES.core.boundaries import GasFeed, PressureReliefVent
from PyOMES.core.phases import R_L_ATM_MOL_K
from PyOMES.control.cv_loops import PHController

print("Imports OK")


### 2.3.1 Parameters

In [ ]:
T_K   = 305.15   # 32 °C
V_TOTAL_L      = 2.0
HEADSPACE_FRAC = 0.20
V_GAS = V_TOTAL_L * HEADSPACE_FRAC        # 0.4 L
V_LIQ = V_TOTAL_L * (1.0 - HEADSPACE_FRAC)  # 1.6 L

MU_MAX      = 0.5     # 1/h
KS_G_L      = 5e-3   # g/L
YIELD       = 0.36    # g biomass / g substrate
PH_SETPOINT = 6.0
KLA_PER_H   = {"O2": 2100.0, "CO2": 1890.0, "NH3": 1890.0}  # CO2/NH3 at 0.9× O2 kLa

S0_G_L     = 20.0   # g/L initial glucose
X0_G_L     = 0.1    # g/L inoculum
NH4CL_G_L  = 5.0    # g/L NH4Cl
KH2PO4_G_L = 30.0   # g/L KH2PO4

TAU_H   = 24.0
N_STEPS = 1000

print(f"V_liq = {V_LIQ:.1f} L   V_gas = {V_GAS:.1f} L   T = {T_K:.2f} K")
print(f"kLa(O2) = {KLA_PER_H['O2']:.0f} /h   vvm = 1   pH setpoint = {PH_SETPOINT}")
print(f"NH4Cl:  {NH4CL_G_L} g/L  ->  {NH4CL_G_L/float(NH4CL.MW)*1000:.1f} mmol/L NH4+ and Cl-")
print(f"KH2PO4: {KH2PO4_G_L} g/L  ->  {KH2PO4_G_L/float(KH2PO4_SALT.MW)*1000:.1f} mmol/L total phosphate")

### 2.3.2 Reaction system

In [ ]:
GLUCOSE = Species(id="Glucose", atoms={"C":6,"H":12,"O":6},               charge=0)
# CHNO biomass formula; MW auto-computed from atoms (~24.694 g/mol)
PEKILO  = Species(id="PEKILO",  atoms={"C":1,"H":1.61,"O":0.56,"N":0.15}, charge=0)

rxn_growth = ReactionBuilder.monod_aerobic_growth(
    substrate=GLUCOSE, biomass=PEKILO,
    mu_max_per_h=MU_MAX, Ks_gL=KS_G_L, yield_gX_gS=YIELD,
    Ko2_gL=0.2e-3,
    balance="CHNO",
    label="growth_on_Glucose",
)

# Stoichiometric O2 coefficient for use in validation
nu_O2_act = abs(next(e.coefficient for e in rxn_growth.stoichiometry
                     if e.species.id == "O2"))
print(f"PEKILO MW (auto): {float(PEKILO.MW):.3f} g/mol")
print(f"nu_O2 from stoichiometry: {nu_O2_act:.4f} mol O2 / mol glucose")
rxn_growth.show_stoichiometry()


In [ ]:
RELEVANT = {
    "eq_water", "eq_CO2", "eq_NH4",
    "eq_phosphate_1", "eq_phosphate_2", "eq_phosphate_3",
}
db_rxns = [r for r in BIOPROCESS_BASIC.reactions if r.label in RELEVANT]

rxn_system = ReactionSystem(
    [rxn_growth] + db_rxns,
    label="batch_fermenter_chemistry",
)

print(f"ReactionSystem: {len(rxn_system.kinetic_reactions)} kinetic, "
      f"{len(rxn_system.single_phase_equilibria)} single-phase equilibria")


Buffer speciation at the operating temperature confirms the dominant forms at pH 6 — H₂PO₄⁻
for phosphate (well below the pKₐ 7.2 buffer pair) and CO₂(aq) for inorganic carbon (below pKₐ 6.35).
The pH setpoint is marked with a dashed line.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

_, ax_p = rxn_system.plot_speciation("H3PO4", T_K=T_K, ax=axes[0])
ax_p.axvline(PH_SETPOINT, color="tab:gray", ls="--", lw=1.2,
             label=f"pH setpoint ({PH_SETPOINT})")
ax_p.legend(fontsize=8)

_, ax_c = rxn_system.plot_speciation("CO2", T_K=T_K, ax=axes[1])
ax_c.axvline(PH_SETPOINT, color="tab:gray", ls="--", lw=1.2,
             label=f"pH setpoint ({PH_SETPOINT})")
ax_c.legend(fontsize=8)

fig.suptitle(f"Buffer speciation at T = {T_K:.0f} K (32 degC)", fontsize=11)
fig.tight_layout()
plt.show()


### 2.3.3 Gas-liquid mass transfer

Phases are initialised at Henry-law equilibrium for dissolved gases. `KineticTransferModel`
is attached for O₂, CO₂, and NH₃; `EquilibriumTransferModel` for N₂.

In [ ]:
n_total_gas = V_GAS / (R_L_ATM_MOL_K * T_K)

gas = GasPhase(
    n_mol={"O2": n_total_gas*0.2095, "CO2": n_total_gas*0.0004,
           "N2": n_total_gas*0.7901, "NH3": 0.0},
    V_L=V_GAS, T_K=T_K,
)

def _henry_n(sp):
    pm = AD_BASIC.partition_models[sp]
    return pm.H_ref * gas.p_atm.get(sp, 0.0) * 101325 / 1000 * V_LIQ

liquid = LiquidPhase(
    n_mol={
        GLUCOSE.id:     (S0_G_L     / float(GLUCOSE.MW))        * V_LIQ,
        PEKILO.id:      (X0_G_L     / float(PEKILO.MW))         * V_LIQ,
        NH4_PLUS.id:    (NH4CL_G_L  / float(NH4CL.MW))         * V_LIQ,
        CHLORIDE.id:    (NH4CL_G_L  / float(NH4CL.MW))         * V_LIQ,
        K_PLUS.id:      (KH2PO4_G_L / float(KH2PO4_SALT.MW))  * V_LIQ,
        H2PO4_MINUS.id: (KH2PO4_G_L / float(KH2PO4_SALT.MW))  * V_LIQ,
        HPO4_2MINUS.id: 0.0,  H3PO4.id: 0.0,  PO4_3MINUS.id: 0.0,
        NA_PLUS.id:     0.0,
        "O2":    _henry_n("O2"),
        "CO2":   _henry_n("CO2"),
        "N2":    _henry_n("N2"),
        "NH3":   0.0,
        "HCO3-": 0.0,  "CO3--": 0.0,  "OH-": 0.0,  "H+": 1e-7 * V_LIQ,
    },
    V_L=V_LIQ, T_K=T_K,
)

transfer_models = {
    "O2":  KineticTransferModel(AD_BASIC.partition_models["O2"],
               k_transfer=KLA_PER_H["O2"]),
    "CO2": KineticTransferModel(AD_BASIC.partition_models["CO2"],
               k_transfer=KLA_PER_H["CO2"], transfer_basis="molecular"),
    "NH3": KineticTransferModel(AD_BASIC.partition_models["NH3"],
               k_transfer=KLA_PER_H["NH3"], transfer_basis="molecular"),
    "N2":  EquilibriumTransferModel(AD_BASIC.partition_models["N2"]),
}

cv = ControlVolume(
    phases={"gas": gas, "liquid": liquid},
    transfer_models=transfer_models,
    reaction_system=rxn_system,
    label="batch_fermenter",
)
print("CV assembled —", len(cv.boundaries), "boundaries (no sparger yet)")


### 2.3.4 Initial pH correction

The KH₂PO₄-dominated medium has a native equilibrium pH near 4.7 (H₂PO₄⁻ is acidic).
`equilibrate_to_pH` adds the minimum NaOH to reach pH 6 before the simulation starts.

In [ ]:
n_corr = cv.equilibrate_to_pH("NaOH", PH_SETPOINT)


### 2.3.5 Boundaries and controller

A `GasFeed` supplies air at 1 vvm; a `PressureReliefVent` clips headspace pressure at 1.1 atm.
A proportional `PHController` doses H₃PO₄ (acid) or NaOH (base) to hold pH 6.0, capped at
0.05 mol/L/h.

In [ ]:
cv.boundaries.append(GasFeed(
    vvm_min=1.0, y={"O2": 0.21, "N2": 0.79},
    P_inlet_atm=1.0, phase_key="gas", liquid_phase_key="liquid",
    label="air_sparge",
))
cv.boundaries.append(PressureReliefVent(P_set_atm=1.10, mode="instant"))

ph_ctrl = PHController(
    setpoint=PH_SETPOINT, Kp=0.5, Ki=0.0,
    chemical_id="H3PO4", base_chemical_id="NaOH",
    max_add_molL_hr=0.05,
)

print("Boundaries:", [b.label for b in cv.boundaries])
print("pH controller setpoint:", ph_ctrl.setpoint)


### 2.3.6 Simulation run

In [ ]:
sim    = Simulation(cvs={"main": cv}, controllers=[ph_ctrl], label="batch_fermenter_sim")
result = sim.run(tau_h=TAU_H, n_steps=N_STEPS)
print(f"Done in {result.runtime_s:.2f} s")


### 2.3.7 Results

Each of the four design-basis predictions is checked against the simulation.

#### 2.3.7.1 OTR vs OUR balance

**Predictions 1 and 2**: OTR exceeds OUR throughout the run; DO stays above zero;
substrate is the binding constraint.

In [ ]:
liq     = result.liquid_mol["main"]
gas_mol = result.gas_mol["main"]
t       = result.t_h
MW_S    = float(GLUCOSE.MW)
MW_X    = float(PEKILO.MW)

# Headspace O2 partial pressure at each timestep
n_gas_tot = sum(gas_mol[sp] for sp in gas_mol)
P_tot_t   = n_gas_tot * R_L_ATM_MOL_K * T_K / V_GAS   # atm
y_O2_t    = gas_mol["O2"] / n_gas_tot
P_O2_t    = y_O2_t * P_tot_t

# Henry equilibrium dissolved O2 (mol/L)
pm_O2    = AD_BASIC.partition_models["O2"]
C_star_t = pm_O2.H_ref * P_O2_t * 101325 / 1000
C_O2_t   = liq["O2"] / V_LIQ

# OTR and OUR (mol/h)
OTR_t = KLA_PER_H["O2"] * (C_star_t - C_O2_t) * V_LIQ
S_gL  = liq["Glucose"] / V_LIQ * MW_S
X_gL  = liq["PEKILO"]  / V_LIQ * MW_X
mu_t  = MU_MAX * S_gL / (KS_G_L + S_gL)
OUR_t = nu_O2_act * (mu_t / YIELD * X_gL / MW_S * V_LIQ)

# DO percent saturation
DO_pct = C_O2_t / np.maximum(C_star_t, 1e-30) * 100

print("=" * 50)
print("OTR / OUR validation")
print("=" * 50)
print(f"\nPeak OTR  : {OTR_t.max()*1000:.1f} mmol/h")
print(f"Peak OUR  : {OUR_t.max()*1000:.1f} mmol/h")
print(f"Min OTR/OUR ratio: {(OTR_t / np.maximum(OUR_t, 1e-30)).min():.2f}")
status1 = "PASS" if (OTR_t >= OUR_t * 0.9).all() else "FAIL"
print(f"OTR >= OUR throughout: {status1}")
print(f"\nMinimum DO: {DO_pct.min():.1f}%  (design basis: > 0%)")
status2 = "PASS" if DO_pct.min() > 0.5 else "FAIL"
print(f"DO > 0 throughout: {status2}")


#### 2.3.7.2 Yield and endpoint biomass

**Predictions 3 and 4**: endpoint biomass ≈ 0.53 g/L; pH tracks the setpoint.

In [ ]:
C_S0 = liq["Glucose"][0]  / V_LIQ * MW_S
C_Sf = liq["Glucose"][-1] / V_LIQ * MW_S
C_X0 = liq["PEKILO"][0]   / V_LIQ * MW_X
C_Xf = liq["PEKILO"][-1]  / V_LIQ * MW_X
delta_S = C_S0 - C_Sf
delta_X = C_Xf - C_X0
Y_obs   = delta_X / delta_S if delta_S > 1e-6 else float("nan")
X_pred  = YIELD * S0_G_L + X0_G_L

print("Prediction 3 — endpoint biomass:")
print(f"  Predicted X_f : {X_pred:.3f} g/L")
print(f"  Simulated X_f : {C_Xf:.3f} g/L")
err3 = abs(C_Xf - X_pred) / X_pred * 100
print(f"  Error         : {err3:.1f}%  [{('PASS' if err3 < 5 else 'CHECK')}]")
print(f"  Observed yield dX/dS = {Y_obs:.3f}  (Y = {YIELD:.3f})")

pH     = result.pH["main"]
ph_val = pH[np.isfinite(pH)]
max_dev = np.abs(ph_val - PH_SETPOINT).max()
print(f"\nPrediction 4 — pH tracking:")
print(f"  pH range  : {ph_val.min():.3f} - {ph_val.max():.3f}")
print(f"  Max deviation from setpoint: {max_dev:.3f}")
print(f"  [{('PASS' if max_dev < 0.5 else 'CHECK')}]")


#### 2.3.7.3 Time-series plots

Annotations mark the design-basis reference lines for direct comparison.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle(
    f"Batch Fermenter — V_liq = {V_LIQ:.1f} L, kLa(O2) = {KLA_PER_H['O2']:.0f} /h,"
    f" S0 = {S0_G_L} g/L",
    fontsize=12,
)

# OTR vs OUR
ax = axes[0, 0]
ax.plot(t, OTR_t * 1000, color="tab:blue",   label="OTR")
ax.plot(t, OUR_t * 1000, color="tab:orange", label="OUR")
ax.fill_between(t, OUR_t*1000, OTR_t*1000,
                where=(OTR_t >= OUR_t), alpha=0.12, color="tab:blue",
                label="OTR surplus")
ax.set(xlabel="Time (h)", ylabel="mmol O2/h", title="OTR vs OUR")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Dissolved O2
ax = axes[0, 1]
ax.plot(t, DO_pct, color="tab:blue")
ax.axhline(50, ls="--", color="gray", lw=1, label="50% (design basis)")
ax.set(xlabel="Time (h)", ylabel="DO (% saturation)", title="Dissolved Oxygen")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# Substrate and biomass
ax  = axes[1, 0]
C_S = liq["Glucose"] / V_LIQ * MW_S
C_X = liq["PEKILO"]  / V_LIQ * MW_X
ax.plot(t, C_S, color="tab:orange", label="Glucose (g/L)")
ax.plot(t, C_X, color="tab:green",  label="PEKILO (g/L)")
ax.axhline(X_pred, ls=":", color="tab:green", lw=1.2,
           label=f"Predicted X_f = {X_pred:.2f} g/L")
ax.set(xlabel="Time (h)", ylabel="Concentration (g/L)",
       title="Substrate and Biomass")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# pH
ax = axes[1, 1]
ax.plot(t, pH, color="tab:red", label="pH")
ax.axhline(PH_SETPOINT, ls="--", color="gray", lw=1.2,
           label=f"Setpoint ({PH_SETPOINT})")
ax.set(xlabel="Time (h)", ylabel="pH", title="pH")
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()


### 2.3.8 Summary

At kLa = 2100 /h and S₀ = 20 g/L the fermentation is substrate-limited throughout: OTR exceeds
OUR throughout the run, DO stays well above zero, and the endpoint biomass of ~7.3 g/L matches
the yield-based prediction. The proportional pH controller holds pH within 0.1 units of the 6.0
setpoint; the phosphate buffer (H₂PO₄⁻/HPO₄²⁻, pKₐ 7.2) absorbs the CO₂-driven acid load
between controller steps, keeping dose rates low.

The What-If Analysis below maps the sensitivity to the three most uncertain design parameters.

## 2.4 What-If Analysis

Three single-variable sweeps explore the key design parameters, all other variables held at the §2.3
base values (T = 32 °C, V_liq = 1.6 L, S₀ = 20 g/L, kLa = 2100 /h, pH 6.0, 1 vvm):

| Sweep | Varied | Fixed | Output |
|---|---|---|---|
| 1 | kLa (200 – 4000 /h) | S₀ = 20 g/L, pH 6.0 | Minimum DO (% saturation) |
| 2 | Initial glucose S₀ (2 – 40 g/L) | kLa = 2100 /h, pH 6.0 | Endpoint biomass (g/L) |
| 3 | pH setpoint (5.0 – 7.5) | kLa = 2100 /h, S₀ = 20 g/L | NaOH dosed by controller (mmol/L) |

Sweep 3 measures the Na⁺ increment during the run (post-pH-correction to end): the cumulative
base that the `PHController` must dose to counteract CO₂-driven acidification.

In [ ]:
import warnings

# ── Sweep helpers ─────────────────────────────────────────────────────────────
def _make_cv_batch(S0_gL, kla_o2, pH_sp):
    _n_g = V_GAS / (R_L_ATM_MOL_K * T_K)
    _gas = GasPhase(
        n_mol={"O2": _n_g*0.2095, "CO2": _n_g*0.0004,
               "N2": _n_g*0.7901, "NH3": 0.0},
        V_L=V_GAS, T_K=T_K,
    )
    _hn = lambda sp: (AD_BASIC.partition_models[sp].H_ref
                      * _gas.p_atm.get(sp, 0.0) * 101325 / 1000 * V_LIQ)
    _n_p = (KH2PO4_G_L / float(KH2PO4_SALT.MW)) * V_LIQ
    _liq = LiquidPhase(
        n_mol={
            GLUCOSE.id:     (S0_gL     / float(GLUCOSE.MW))       * V_LIQ,
            PEKILO.id:      (X0_G_L    / float(PEKILO.MW))        * V_LIQ,
            NH4_PLUS.id:    (NH4CL_G_L / float(NH4CL.MW))        * V_LIQ,
            CHLORIDE.id:    (NH4CL_G_L / float(NH4CL.MW))        * V_LIQ,
            K_PLUS.id: _n_p, H2PO4_MINUS.id: _n_p,
            HPO4_2MINUS.id: 0.0, H3PO4.id: 0.0, PO4_3MINUS.id: 0.0,
            NA_PLUS.id: 0.0,
            "O2": _hn("O2"), "CO2": _hn("CO2"), "N2": _hn("N2"), "NH3": 0.0,
            "HCO3-": 0.0, "CO3--": 0.0, "OH-": 0.0, "H+": 1e-7 * V_LIQ,
        },
        V_L=V_LIQ, T_K=T_K,
    )
    _kla = {sp: kla_o2 * (1.0 if sp == "O2" else 0.9) for sp in ("O2", "CO2", "NH3")}
    _tm = {
        "O2":  KineticTransferModel(AD_BASIC.partition_models["O2"],
                   k_transfer=_kla["O2"]),
        "CO2": KineticTransferModel(AD_BASIC.partition_models["CO2"],
                   k_transfer=_kla["CO2"], transfer_basis="molecular"),
        "NH3": KineticTransferModel(AD_BASIC.partition_models["NH3"],
                   k_transfer=_kla["NH3"], transfer_basis="molecular"),
        "N2":  EquilibriumTransferModel(AD_BASIC.partition_models["N2"]),
    }
    _cv = ControlVolume(
        phases={"gas": _gas, "liquid": _liq},
        transfer_models=_tm,
        reaction_system=rxn_system,
        label="sw",
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _cv.equilibrate_to_pH("NaOH", pH_sp)
    _cv.boundaries.append(GasFeed(vvm_min=1.0, y={"O2": 0.21, "N2": 0.79},
                                   P_inlet_atm=1.0, phase_key="gas",
                                   liquid_phase_key="liquid", label="air"))
    _cv.boundaries.append(PressureReliefVent(P_set_atm=1.10, mode="instant"))
    return _cv

def _run_sw(cv_, pH_sp=6.0):
    _ctrl = PHController(setpoint=pH_sp, Kp=0.5, Ki=0.0,
                         chemical_id="H3PO4", base_chemical_id="NaOH",
                         max_add_molL_hr=0.05)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return Simulation(cvs={"main": cv_}, controllers=[_ctrl], label="sw").run(
            tau_h=TAU_H, n_steps=N_STEPS // 2)

# ── Sweep 1: kLa vs minimum DO ────────────────────────────────────────────────
# Range spans from well below the critical kLa (~2042 /h) to comfortably above.
KLA_vals = [200, 500, 800, 1200, 1600, 2000, 2100, 3000, 4000]
min_DO_1 = []
print("Sweep 1 — kLa vs minimum DO")
for kla in KLA_vals:
    cv_ = _make_cv_batch(S0_G_L, kla, PH_SETPOINT)
    res = _run_sw(cv_, PH_SETPOINT)
    _liq = res.liquid_mol["main"]; _gas = res.gas_mol["main"]
    _n_g = sum(_gas[sp] for sp in _gas)
    _P_O2 = _gas["O2"] / _n_g * (_n_g * R_L_ATM_MOL_K * T_K / V_GAS)
    _Cst  = AD_BASIC.partition_models["O2"].H_ref * _P_O2 * 101325 / 1000
    _DO   = _liq["O2"] / V_LIQ / np.maximum(_Cst, 1e-30) * 100
    min_DO_1.append(float(_DO.min()))
    print(f"  kLa={kla:.0f}  minDO={min_DO_1[-1]:.1f}%")

# ── Sweep 2: S0 vs endpoint biomass ──────────────────────────────────────────
# Range spans from low loading to above the design S0 = 20 g/L.
S0_vals = [2.0, 5.0, 10.0, 15.0, 20.0, 25.0, 30.0, 40.0]
Xf_2    = []
print(f"\nSweep 2 — initial substrate vs endpoint biomass (kLa = {KLA_PER_H['O2']:.0f})")
for S0 in S0_vals:
    cv_ = _make_cv_batch(S0, KLA_PER_H["O2"], PH_SETPOINT)
    res = _run_sw(cv_, PH_SETPOINT)
    Xf_2.append(float(res.liquid_mol["main"]["PEKILO"][-1] / V_LIQ * float(PEKILO.MW)))
    print(f"  S0={S0:.1f}  Xf={Xf_2[-1]:.3f} g/L")

# ── Sweep 3: pH setpoint vs total NaOH dose ──────────────────────────────────
PH_vals  = [5.0, 5.5, 5.75, 6.0, 6.25, 6.5, 7.0, 7.5]
naoh_3   = []
print(f"\nSweep 3 — pH setpoint vs controller NaOH dose (kLa = {KLA_PER_H['O2']:.0f})")
for pH_sp in PH_vals:
    cv_ = _make_cv_batch(S0_G_L, KLA_PER_H["O2"], pH_sp)
    res = _run_sw(cv_, pH_sp)
    _liq = res.liquid_mol["main"]
    _naoh = (_liq[NA_PLUS.id][-1] - _liq[NA_PLUS.id][0]) / V_LIQ * 1000  # mmol/L
    naoh_3.append(float(_naoh))
    print(f"  pH={pH_sp:.2f}  NaOH_ctrl={naoh_3[-1]:.2f} mmol/L")

print("\nAll sweeps complete.")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(8, 13))

# Sweep 1: kLa vs minimum DO
ax1 = axes[0]
ax1.plot(KLA_vals, min_DO_1, "o-", color="tab:blue")
ax1.axhline(0, color="gray", ls="--", lw=0.8, alpha=0.7)
ax1.axvline(KLA_PER_H["O2"], color="tab:green", ls=":", lw=1.2,
            label=f"Design-basis kLa = {KLA_PER_H['O2']:.0f} /h")
ax1.set_xlabel("kLa (O2) (/h)")
ax1.set_ylabel("Minimum DO (% saturation)")
ax1.set_title(f"(1)  Mass transfer   [S0 = {S0_G_L} g/L,  pH {PH_SETPOINT}]", fontsize=10)
ax1.legend(fontsize=9); ax1.grid(True, alpha=0.3)

# Sweep 2: S0 vs endpoint biomass
ax2  = axes[1]
ax2b = ax2.twinx()
Xf_pred_2 = [YIELD * s + X0_G_L for s in S0_vals]
l1, = ax2.plot( S0_vals, Xf_2,      "o-",  color="tab:green", label="Simulated X_f")
l2, = ax2.plot( S0_vals, Xf_pred_2, "k--", lw=0.9, label=f"Y*S0 + X0  (Y={YIELD})")
conv_2 = [(Xf - X0_G_L) / (YIELD * s + 1e-9) * 100 for Xf, s in zip(Xf_2, S0_vals)]
l3, = ax2b.plot(S0_vals, conv_2, "s--", color="tab:orange", label="Biomass yield conv. (%)")
ax2.set_xlabel("Initial glucose S0 (g/L)")
ax2.set_ylabel("Endpoint biomass (g/L)", color="tab:green")
ax2b.set_ylabel("Yield conversion (%)", color="tab:orange")
ax2.set_title(f"(2)  Substrate loading   [kLa = {KLA_PER_H['O2']:.0f} /h,  pH {PH_SETPOINT}]",
              fontsize=10)
ax2.legend(handles=[l1, l2, l3], fontsize=8, loc="lower right")
ax2.grid(True, alpha=0.3)

# Sweep 3: pH setpoint vs NaOH dose
ax3 = axes[2]
ax3.plot(PH_vals, naoh_3, "o-", color="tab:red")
ax3.axvline(PH_SETPOINT, color="tab:green", ls=":", lw=1.2,
            label=f"Design-basis pH {PH_SETPOINT}")
ax3.axhline(0, color="gray", ls="--", lw=0.8, alpha=0.7)
ax3.set_xlabel("pH setpoint")
ax3.set_ylabel("NaOH dosed by controller (mmol/L)")
ax3.set_title(f"(3)  pH setpoint   [kLa = {KLA_PER_H['O2']:.0f} /h,  S0 = {S0_G_L} g/L]",
              fontsize=10)
ax3.legend(fontsize=9); ax3.grid(True, alpha=0.3)

fig.suptitle("What-if analysis — 2 L batch fermenter, T = 32 degC, 1 vvm", fontsize=11)
fig.tight_layout(h_pad=3.5)
plt.show()


### 2.4.1 Summary

**Sweep 1 — kLa:** Minimum DO falls steeply below kLa ≈ 2000 /h: at the high peak biomass
(X_peak ≈ 7.3 g/L) driven by S₀ = 20 g/L, the critical mass-transfer requirement is far higher
than in a low-substrate design. Above ~2100 /h the curve flattens and DO stays well above the
half-saturation constant $K_{O_2}$ = 0.2 × 10⁻³ g/L, confirming the design-basis kLa delivers
the required 1.5× safety margin.

**Sweep 2 — substrate loading:** Endpoint biomass rises linearly with S₀ up to ~20 g/L where
the oxygen budget is still adequate at kLa = 2100 /h. At higher loadings the peak OUR starts to
approach the maximum OTR and the observed yield begins to trail the theoretical Y = 0.36,
signalling the onset of O₂ limitation.

**Sweep 3 — pH setpoint:** Controller NaOH demand rises non-linearly with pH setpoint. Higher
setpoints require more base to counteract CO₂-driven acidification; the steep rise above pH 6.5
reflects the approach to the carbonate p$K_a$ (6.35), where HCO₃⁻ accumulates and NaOH is
consumed rapidly to maintain the target. pH 6.0 sits in the low-slope region, giving adequate
pH control at minimal base cost.